#### Classification finetuning 

For our classification finetuning we will be using 2 Datasets, and I will be giving one as practice for anyone interested

In [2]:
## 1. SPAM Classifier
## 2. Toxic Comment classification
## 3. Amazon Reviews Polarity(sentiment Analysis)
## Practice: 4. Emotional Dataset 
## Practice: IMDB Dataset Classification
## Project of this part: Recipe or food makig description with a perfect UI

### Preparing the Dataset, Finetuning, and Evaluation

We start with downloading, inspecting, and preparing the dataset that we'll use to finetune the model. The process runs through three broad stages:

1. **Dataset preparation** — download, preprocess, and create data loaders
2. **Model setup** — initialize, load pretrained weights, and modify for finetuning
3. **Finetuning and usage** — train, evaluate, and use the model on new data

![Finetuning Stages](classification_finetuning_pipeline.png)

## What We're Doing in This Section

- We're building the dataset that we'll actually use to finetune the GPT model for classification.
- Specifically, this is SMS text messages labeled as either **spam** or **ham** (not spam) — the goal is to teach the model to tell them apart.
- The very first step is getting the raw data onto disk: downloading the zip file from the source and extracting it, so we have the actual text messages + labels to work with.

## Step 1: Download and Unzip the Raw Dataset

We're finetuning the LLM to distinguish **spam** vs. **ham** (non-spam) SMS messages.

Before anything else, we need the raw data on disk — download the zip file from the source and extract it, so we have the actual text messages and their labels ready to load.

In [3]:
import os
os.chdir("../..")

In [4]:
pwd

'c:\\Users\\user\\Documents\\AI-EngineeringRoadmap\\llm-from-scratch'

In [5]:
from data_loader import download_and_unzip_spam_data
import importlib.util
from pathlib import Path

module_path = Path("02_tiny_llm/data/tokenizer_utils.py") 
spec = importlib.util.spec_from_file_location("tokenizer_utils", module_path)
tokenizer_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tokenizer_utils)

get_tokenizer = tokenizer_utils.get_tokenizer

data_path = download_and_unzip_spam_data()

import pandas as pd
df = pd.read_csv(data_path, sep="\t", header=None, names=["Label", "Text"])
print(df.head())
print(df["Label"].value_counts())

Trying primary (UCI): https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip
Saved to sms_spam_collection.zip
Extracting...
File saved as sms_spam_collection\SMSSpamCollection.tsv
  Label                                               Text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
Label
ham     4825
spam     747
Name: count, dtype: int64


## Step 2: Load the Dataset and Inspect Class Balance

With the raw file extracted, we load it into a pandas DataFrame and take a first look at what we're working with — the message text, its label, and how many examples we have of each class.

This step matters because it's where we discover whether the dataset is balanced. If one class heavily outnumbers the other (as it does here — far more ham than spam), we'll need to address that before training, or the model can learn to just always predict the majority class and still look "accurate."

In [6]:
df.head()

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
df.shape

(5572, 2)

In [8]:
df.Label.value_counts()

Label
ham     4825
spam     747
Name: count, dtype: int64

In [9]:
### Imbalance Dataset: So Accuracy is not a good one, so we will be using Recall Precision, and also F1-score

In [10]:
df[df['Label'] == "spam"].shape[0]

747

## Step 3: Balance the Dataset

Since ham significantly outnumbers spam, we balance the classes before training — otherwise the model can achieve high accuracy just by always predicting "ham," without actually learning to recognize spam.

We do this by undersampling: randomly reducing the ham examples down to match the spam count, so both classes are equally represented. This keeps every example real (no duplication or synthetic data), at the cost of discarding some ham messages we don't strictly need.

In [11]:
def create_balanced_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Balance a spam/ham dataframe by undersampling the majority class (ham)
    to match the minority class (spam) count.
    """
    num_spam = df[df["Label"] == "spam"].shape[0]

    # Randomly sample ham rows down to the same count as spam
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)

    # Combine spam with the downsampled ham
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])

    return balanced_df


balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [12]:
balanced_df.head()

,Label,Text
4307,ham,Awww dat is sweet! We can think of something t...
4138,ham,Just got to &lt;#&gt;
4831,ham,"The word ""Checkmate"" in chess comes from the P..."
4461,ham,This is wishing you a great day. Moji told me ...
5440,ham,Thank you. do you generally date the brothas?


## Step 4: Convert Labels to Integers

The model needs numeric labels to train on, not text. We map the string labels to integers — `"ham"` → `0` and `"spam"` → `1` — so they can be used directly as classification targets.

In [13]:
balanced_df['Label'] = balanced_df['Label'].map({"ham": 0, "spam": 1})
### The output head of the GPT2 model will now be a binary class head
balanced_df.head()

,Label,Text
4307,0,Awww dat is sweet! We can think of something t...
4138,0,Just got to &lt;#&gt;
4831,0,"The word ""Checkmate"" in chess comes from the P..."
4461,0,This is wishing you a great day. Moji told me ...
5440,0,Thank you. do you generally date the brothas?


## Step 5: Split into Train, Validation, and Test Sets

Before training, we shuffle the balanced dataset and split it into three parts: a training set to learn from, a validation set to check progress and tune decisions during training, and a held-out test set to evaluate final performance on data the model has never seen.

We shuffle first so ham and spam examples are mixed throughout, then split — commonly using a ratio like 70% train, 10% validation, and 20% test.

In [14]:
def random_split(df: pd.DataFrame, train_frac: float, validation_frac: float):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)  # shuffle

    train_end = int(len(df) * train_frac)
    val_end = train_end + int(len(df) * validation_frac)

    train_df = df[:train_end]
    val_df = df[train_end:val_end]
    test_df = df[val_end:]

    return train_df, val_df, test_df


train_df, val_df, test_df = random_split(balanced_df, train_frac=0.7, validation_frac=0.1)
# remaining 0.2 becomes test

print(len(train_df), len(val_df), len(test_df))

1045 149 300


## Step 6: Save the Splits to Disk

We save the train, validation, and test splits as separate CSV files. This keeps the split fixed and reproducible — we tokenize and load from these saved files going forward, rather than re-shuffling and re-splitting every time we rerun the notebook.

In [15]:
train_df.to_csv("sms_spam_collection/train.csv", index=False)
test_df.to_csv("sms_spam_collection/test.csv", index=False)
val_df.to_csv("sms_spam_collection/val.csv", index=False)

In [16]:
train = pd.read_csv("sms_spam_collection/train.csv")
test = pd.read_csv("sms_spam_collection/test.csv")
validation = pd.read_csv("sms_spam_collection/val.csv")

In [17]:
train.shape

(1045, 2)

In [18]:
test.shape

(300, 2)

In [19]:
validation.shape

(149, 2)

## Step 7: Tokenize and Build the Dataset

With clean, split, saved data in hand, we tokenize the text and wrap each split in a `Dataset` class. Every message gets encoded into token IDs and padded to a consistent length, so the batches can be fed into the model.

We also decide on a padding token here — reusing the `<|endoftext|>` token works well, since GPT-2 doesn't have a dedicated pad token of its own.

In [20]:
tokenizer = get_tokenizer()

In [21]:
train["token_ids"] = df["Text"].apply(lambda x: tokenizer.encode(x))

In [22]:
train["length_of_tokens"] = train["token_ids"].apply(lambda x: len(x))

In [23]:
train.head()

,Label,Text,token_ids,length_of_tokens
0,0,Dude how do you like the buff wind.,"[5247, 1566, 8174, 506, 966, 11, 7165, 492, 14...",28
1,0,Tessy..pls do me a favor. Pls convey my birthd...,"[18690, 26371, 986, 449, 5730, 266, 361, 334, ...",11
2,1,Reminder: You have not downloaded the content ...,"[11146, 5726, 287, 362, 257, 266, 74, 306, 552...",50
3,1,Got what it takes 2 take part in the WRC Rally...,"[52, 12574, 910, 523, 1903, 3076, 986, 471, 26...",13
4,1,"Shop till u Drop, IS IT YOU, either 10K, 5K, £...","[45, 993, 314, 836, 470, 892, 339, 2925, 284, ...",17


Each row of the data have varying number of token, which should be, so we willl have to take care of that

In [24]:
tokenizer.decode([50256])

'<|endoftext|>'

In [25]:
import torch

In [26]:
from torch.utils.data import Dataset, DataLoader
class SpamDataset(Dataset):
    def __init__(self, file_path: str, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(file_path)

        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data['Text']
        ]
        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            ## Truncate if max_length was passed
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]
        # pad everything to max_length
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

        self.labels = self.data["Label"].tolist()
  
    def _longest_encoded_length(self):
        return max(len(encoded_text) for encoded_text in self.encoded_texts)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.encoded_texts[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )

In [27]:
# Test the SpamDataset class

train_dataset = SpamDataset("sms_spam_collection/train.csv", tokenizer)
val_dataset = SpamDataset("sms_spam_collection/val.csv", tokenizer)
test_dataset = SpamDataset("sms_spam_collection/test.csv", tokenizer)

In [28]:
input_ids, target = train_dataset[0]
print(f"The input_id: {input_ids}\n")
print(f"The target: {target}")

The input_id: tensor([   35,  2507,   703,   466,   345,   588,   262,  6940,  2344,    13,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256])

The target: 0


In [29]:
train_dataset._longest_encoded_length()

120

In [30]:
len(train_dataset[0][0])

120

## Step 8: Wrap the Datasets in DataLoaders

With the `Dataset` objects built for train, validation, and test, we wrap each one in a `DataLoader`. This handles batching, shuffling the training data each epoch, and feeding fixed-size batches into the model during training and evaluation.

We typically shuffle the training loader but not the validation/test loaders, since randomizing order only matters while the model is actively learning.

In [31]:
# Reasonable defaults — batch_size=8 is a good starting point for a small
# classification finetune on CPU/single GPU; bump it up if you have more
# memory available and want faster epochs.
batch_size = 8
num_workers = 0  # increase (e.g. 2-4) if you're on Linux/Mac and want faster loading

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,       # randomize order each epoch — important for training
    drop_last=True,     # drop the last incomplete batch, keeps batch shapes consistent
    num_workers=num_workers,
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,       # no need to shuffle — we're just evaluating
    drop_last=False,     # keep every example, even the last partial batch
    num_workers=num_workers,
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)

In [32]:
print(f"{len(train_dataloader)} training batches")
print(f"{len(val_dataloader)} validation batches")
print(f"{len(test_dataloader)} test batches")

# Peek at one batch
input_batch, target_batch = next(iter(train_dataloader))
print("Input batch shape:", input_batch.shape)   # e.g. [8, max_length]
print("Target batch shape:", target_batch.shape) # e.g. [8]

130 training batches
19 validation batches
38 test batches
Input batch shape: torch.Size([8, 120])
Target batch shape: torch.Size([8])


## Step 9: Prepare the Model for Classification

Before finetuning, we adapt the pretrained GPT model for this new task. That means swapping out the original output head (which predicts one of 50,257 vocabulary tokens) for a small classification head that outputs just 2 values — one score per class (ham or spam).

We also decide which layers to actually train: commonly, most of the pretrained model is frozen, and only the new classification head (plus maybe the last transformer block) is updated — this trains much faster and works well since the pretrained model already understands language broadly.

#### Initializing a model with pretrained weight as we have done in 02_tiny_llm

In [33]:
pwd

'c:\\Users\\user\\Documents\\AI-EngineeringRoadmap\\llm-from-scratch'

In [34]:
import sys

In [35]:
import sys
from pathlib import Path

# Add the folder CONTAINING "model/" to sys.path — NOT "model" itself
sys.path.append(str(Path("02_tiny_llm").resolve()))

from model.gpt_model import GPTModel
from model.config import GPTConfig

In [36]:
from dataclasses import asdict

GPT_CONFIG_124M = asdict(GPTConfig(50257, 256, 768, 12, 12))
GPT_CONFIG_124M

{'vocab_size': 50257,
 'context_length': 256,
 'emb_dim': 768,
 'n_heads': 12,
 'n_layers': 12,
 'drop_rate': 0.1,
 'qkv_bias': False}

In [37]:
# Define model configurations in a dictionary for compactness
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Copy the base configuration and update with specific model settings
model_name = "gpt2-small (124M)"  # Example model name
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval();

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

Using device: cpu


In [38]:
gpt

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attn): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attn): MultiHeadAttention(
        (W_query): Li

In [39]:
from gpt2_loader import download_and_load_gpt2, load_weights_into_gpt

In [40]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")
load_weights_into_gpt(gpt, params)

In [41]:
gpt

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attn): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attn): MultiHeadAttention(
        (W_query): Li

In [42]:
## Now lets make prediction of of this gpt2loader

def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if top_k:
            top_logits, top_pos = torch.topk(logits, top_k)
            logits = torch.where(
                condition=logits < top_logits[:, -1],
                input=torch.tensor(float("-inf")),
                other=logits
            )

        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if idx_next == eos_id:
            break

        idx = torch.concat((idx, idx_next), dim=1)

    return tokenizer.decode(idx.squeeze().tolist())

def text_to_token_ids(text, tokenizer):
    ids = torch.tensor(tokenizer.encode(text)).unsqueeze(0).to(device)
    return ids

text_to_token_ids("Every Effort moves you", tokenizer)

tensor([[ 6109, 27848,   419,  6100,   345]])

In [43]:
ids = text_to_token_ids("Every Effort moves you", tokenizer)

In [44]:
generate(gpt, ids, 15, context_size=NEW_CONFIG['context_length'], temperature=0.6, top_k=50)

"Every Effort moves you to a higher level of your character, and it's a great way to"

### Swapping the `out_head` for a Binary classification Head

In [45]:
gpt

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attn): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attn): MultiHeadAttention(
        (W_query): Li

In [46]:
gpt.out_head

Linear(in_features=768, out_features=50257, bias=False)

In [47]:
## Making Model Non trainable: Freezing the weights
for param in gpt.parameters():
    param.requires_grad = False

In [48]:
## Now swapping the out_head for a classification head
import torch.nn as nn 

gpt.out_head = nn.Linear(GPT_CONFIG_124M['emb_dim'], 2)

In [49]:
gpt

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attn): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attn): MultiHeadAttention(
        (W_query): Li

In [50]:
total_params = sum(param.numel() for param in gpt.parameters())
trainable_params = sum(param.numel() for param in gpt.parameters() if param.requires_grad == True)

In [51]:
Untrained_params = total_params - trainable_params
Untrained_params 

124439808

In [52]:
trainable_params

1538

In [53]:
## Now lets make the Final LayerNorm, and the last transformer block trainable

for param in gpt.final_norm.parameters():
    param.requires_grad = True 

for param in gpt.trf_blocks[-1].parameters():
    param.requires_grad = True

In [54]:
total_params = sum(param.numel() for param in gpt.parameters())
trainable_params = sum(param.numel() for param in gpt.parameters() if param.requires_grad == True)
Untrained_params = total_params - trainable_params
Untrained_params 

117350400

In [55]:
trainable_params

7090946

In [ ]:
# Implement a class for the classification layer, that allows you to choose how many transformer block you want + 2 last layers
# The class must have generate, train, and also chart plot of time, against the number of trained_layers, and also 
# accuracy(Test, val, train) for number of trainable layers also

# Write in experiments.py, and test in tests folder